In [1]:
import pandas as pd
import numpy as np
from universe import UniverseBuilder
from backtester import BacktestEngine

# Quick test
ub = UniverseBuilder()


In [2]:
print("Running Backtest...")
bt = BacktestEngine(ub)
results = bt.run()
if not results.empty:
    print("\nBacktest Results:")
    print(results.tail())
    metrics = bt.compute_metrics()
    print("\nMetrics:")
    for k, v in metrics.items():
        print(f"{k}: {v}")
else:
    print("No results found.")

Running Backtest...

Backtest Results:
          date    return  n_tickers  cumulative_return
245 2025-10-31  0.004127       1000           3.693780
246 2025-11-28  0.010587       1000           3.732886
247 2025-12-31 -0.000449       1000           3.731209
248 2026-01-30  0.039202       1000           3.877479
249 2026-02-10  0.020657       1000           3.957576

Metrics:
Total Return: 295.76%
Annualized Return: 6.87%
Annualized Vol: 18.52%
Sharpe Ratio: 0.37
Max Drawdown: -54.85%
Count: 250


In [3]:
print(bt.results.columns)

Index(['date', 'return', 'n_tickers', 'cumulative_return'], dtype='str')


In [4]:
spy = bt.compute_spy_benchmark()

merged = bt.results.merge(spy, on="date", how="inner")

print(merged.tail())

          date    return  n_tickers  cumulative_return  spy_return   spy_nav
245 2025-10-31  0.004127       1000           3.693780    0.023837  8.627028
246 2025-11-28  0.010587       1000           3.732886    0.001950  8.643850
247 2025-12-31 -0.000449       1000           3.731209    0.000797  8.650744
248 2026-01-30  0.039202       1000           3.877479    0.014738  8.778236
249 2026-02-10  0.020657       1000           3.957576    0.000217  8.780140


In [5]:
ub.compute_monthly_momentum()

print("Momentum column exists:",
      "momentum" in ub.prices.columns)

print(ub.prices["momentum"].describe())

print("Non-null momentum rows:",
      ub.prices["momentum"].notna().sum())


Momentum column exists: True
count    454368.000000
mean          0.082158
std           0.598085
min          -1.000000
25%          -0.198932
50%           0.013383
75%           0.234094
max           5.000000
Name: momentum, dtype: float64
Non-null momentum rows: 454368


In [6]:
mom_results = bt.run_momentum_strategy()

print(mom_results.tail())

print("Final NAV:", mom_results["nav"].iloc[-1])

KeyError: ['momentum']

In [ ]:
def momentum_metrics(df):

    r = df["return"]
    total_return = (1 + r).prod() - 1
    vol = r.std() * np.sqrt(12)

    years = len(df) / 12
    ann_return = (1 + total_return)**(1/years) - 1
    sharpe = ann_return / vol

    dd = df["nav"] / df["nav"].cummax() - 1

    return {
        "Total Return": total_return,
        "Annual Return": ann_return,
        "Vol": vol,
        "Sharpe": sharpe,
        "Max Drawdown": dd.min()
    }

momentum_metrics(mom_results)
